# Milestone 3 — Measured NCCL/PHB Communication Characterization

Status before execution: **PREPARED_FOR_KAGGLE**. This notebook produces measurements; it does not contain pre-filled GPU results. Use a fresh Kaggle session with Internet enabled and the **T4 x2** accelerator. The primary run leaves NCCL algorithm and protocol selection unforced.

In [ ]:
import json, os, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

WORK = Path('/kaggle/working')
SOURCE = WORK / 'kaggle-vllm-m3-source'
RUNTIME = WORK / 'kaggle-vllm-m3-runtime'
CACHE = WORK / 'kaggle-vllm-cache'
MANIFEST = RUNTIME / 'runtime.json'
REF = 'research/m3-measured-comm-t4-phb-v020'
assert Path('/kaggle').exists(), 'This notebook must run on Kaggle'
assert not SOURCE.exists() and not RUNTIME.exists(), 'Use a fresh session; owned paths already exist'
subprocess.run(['nvidia-smi', '-L'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'kaggle-vllm[hub]==0.2.0'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/kaggle-vllm/kaggle-vllm.git', str(SOURCE)], check=True)
assert '0.2.0' in (SOURCE / 'pyproject.toml').read_text()
RUNTIME.mkdir(); CACHE.mkdir(exist_ok=True)
bootstrap = ['kaggle-vllm', 'bootstrap', '--strict', '--staged', str(RUNTIME/'staged'), '--overlay', str(RUNTIME/'overlay'), '--cache', str(CACHE), '--manifest', str(MANIFEST)]
subprocess.run(bootstrap + ['--dry-run'], check=True)
subprocess.run(bootstrap, check=True)
runtime_data = json.loads(MANIFEST.read_text())
RUN_ENV = dict(os.environ); RUN_ENV.update(runtime_data['runtime_environment'])
RUN_ENV['PYTHONPATH'] = str(SOURCE/'src') + os.pathsep + RUN_ENV['PYTHONPATH']
WHEEL = CACHE / runtime_data['wheel']['filename']
NOTEBOOK = SOURCE / 'kaggle-notebooks/kaggle_vllm_milestone_3_measured_nccl_phb.ipynb'
assert WHEEL.is_file() and NOTEBOOK.is_file()
print('Prepared exact 0.2.0 runtime and clean research checkout:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=SOURCE, text=True).strip())

In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d')
OUTPUT = WORK / f'kaggle-{stamp}-milestone-3-measured-comm'
command = [sys.executable, str(SOURCE/'scripts/kaggle_measured_allreduce.py'), '--output-dir', str(OUTPUT), '--native-wheel', str(WHEEL), '--notebook-source', str(NOTEBOOK), '--m1-dir', str(SOURCE/'artifacts/kaggle-2026-09-01-milestone-1'), '--m2-dir', str(SOURCE/'artifacts/kaggle-2026-09-02-milestone-2'), '--warmups', '20', '--timed-iterations', '100', '--repetitions', '5']
completed = subprocess.run(command, cwd=SOURCE, env=RUN_ENV, check=True)
assert completed.returncode == 0
required = {'M3_RAW_ALLREDUCE.csv','M3_RAW_ALLREDUCE.json','M3_FIT.json','M3_FIT_RESIDUALS.csv','M3_FIT_RESIDUALS.svg','M3_MODEL_VS_OBSERVED.csv','M3_REPORT.md','M3_REPORT.json','M3_ENVIRONMENT.json','M2_SCHEDULER_SIGNALS.json','topology.txt','nccl-info.log','gpu-telemetry.csv','M3_PROVENANCE.json','SHA256SUMS.txt'}
missing = sorted(required - {p.name for p in OUTPUT.iterdir()})
assert not missing, f'Missing required evidence: {missing}'
subprocess.run([sys.executable, '-m', 'kaggle_vllm.research', 'verify-hashes', str(OUTPUT)], env=RUN_ENV, check=True)
print('M3 GPU run produced reviewable evidence at', OUTPUT)

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT)
print('Download this evidence bundle and commit it only after scientific review:', archive)